# GOV-01 V2-E4 — Targeted class-weight training

**Purpose:** continue from the best V2-E3 model and give extra training attention to the three classes selected from validation evidence: **crack**, **pothole**, and **repaired road**.

This is one controlled change. The dataset, labels, train/validation split, model architecture, image size, and protected-test rule remain unchanged. A class weight does not add or alter images: it makes an error on a selected class count more during training.

- Train and validation only; protected test is deliberately never loaded.
- V2-E3 remains unchanged. V2-E4 writes to its own Google Drive folder.
- Run the training cell once. After a disconnect, rerun Cells 1–4, then run Cell 6 and Cell 7; do not retrain.

In [ ]:
# Cell 1 — imports and Google Drive connection
from pathlib import Path
import json
import shutil
import time
import zipfile

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from google.colab import drive, files
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score, precision_recall_fscore_support

drive.mount('/content/drive')
print('TensorFlow:', tf.__version__)
print('GPU available:', bool(tf.config.list_physical_devices('GPU')))

In [ ]:
# Cell 2 — fixed paths and approved V2 class order
DRIVE_ROOT = Path('/content/drive/MyDrive/GOV-01')
ARCHIVE_PATH = DRIVE_ROOT / 'v2' / 'multiclass_final.zip'
E3_CHECKPOINT_PATH = DRIVE_ROOT / 'v2_e3_checkpoint' / 'v2_e3_finetuned_mobilenetv2_best.keras'
PERSISTENT_DIR = DRIVE_ROOT / 'v2_e4_checkpoint'

DATA_DIR = Path('/content/data/processed/v2/multiclass_final')
OUTPUT_DIR = Path('/content/v2_e4_output')
CHECKPOINT_PATH = PERSISTENT_DIR / 'v2_e4_targeted_class_weights_best.keras'
HISTORY_PATH = PERSISTENT_DIR / 'v2_e4_training_history.csv'
SUMMARY_PATH = PERSISTENT_DIR / 'v2_e4_training_summary.json'

CLASS_NAMES = ['crack', 'manhole_cover', 'normal_asphalt', 'pothole', 'repaired_road', 'speed_bump', 'unpaved_road']
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
SEED = 42

if not ARCHIVE_PATH.is_file():
    raise FileNotFoundError(f'Missing V2 dataset ZIP in Google Drive: {ARCHIVE_PATH}')
if not E3_CHECKPOINT_PATH.is_file():
    raise FileNotFoundError(
        f'Missing saved V2-E3 checkpoint: {E3_CHECKPOINT_PATH}\n'
        'Keep the completed V2-E3 checkpoint folder in Google Drive before starting V2-E4.'
    )

print('Dataset ZIP:', ARCHIVE_PATH)
print('Starting model:', E3_CHECKPOINT_PATH)
print('V2-E4 recovery folder:', PERSISTENT_DIR)

In [ ]:
# Cell 3 — extract the persistent dataset into the temporary Colab runtime only when needed.
if not (DATA_DIR / 'train').is_dir() or not (DATA_DIR / 'validation').is_dir():
    DATA_DIR.parent.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(ARCHIVE_PATH) as archive:
        archive.extractall(DATA_DIR.parent)
    print('Extracted dataset to:', DATA_DIR)
else:
    print('Dataset already available in this runtime:', DATA_DIR)

for split_name, expected_count in [('train', 9452), ('validation', 1658)]:
    split_path = DATA_DIR / split_name
    if not split_path.is_dir():
        raise FileNotFoundError(f'Missing required split: {split_path}')
    actual_count = sum(1 for path in split_path.rglob('*') if path.is_file())
    if actual_count != expected_count:
        raise RuntimeError(f'{split_name} has {actual_count} files; expected {expected_count}. Stop and inspect the archive.')
    print(f'{split_name}: {actual_count} images')

if (DATA_DIR / 'protected_test').exists():
    print('Protected test is present in the archive but is intentionally not loaded or evaluated.')

In [ ]:
# Cell 4 — same approved V2 data loaders as E2 and E3
train_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR / 'train', label_mode='int', class_names=CLASS_NAMES,
    image_size=IMAGE_SIZE, batch_size=BATCH_SIZE, shuffle=True, seed=SEED
)
validation_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR / 'validation', label_mode='int', class_names=CLASS_NAMES,
    image_size=IMAGE_SIZE, batch_size=BATCH_SIZE, shuffle=False
)

if train_ds.class_names != CLASS_NAMES or validation_ds.class_names != CLASS_NAMES:
    raise RuntimeError('Class order differs from the approved V2 label order.')

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(AUTOTUNE)
validation_ds = validation_ds.prefetch(AUTOTUNE)
print('Class order:', CLASS_NAMES)
print('Protected test remains unused.')

In [ ]:
# Cell 5 — continue E3 training with targeted class weights. Run this cell once.
# Each selected class is given a bounded extra penalty for mistakes: crack 1.20x; pothole and repaired road 1.60x.
# Other classes remain at 1.00x. These are training priorities, not changes to labels or images.
PERSISTENT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TARGET_CLASS_WEIGHTS = {
    'crack': 1.20,
    'pothole': 1.60,
    'repaired_road': 1.60,
}
class_weight = {
    index: TARGET_CLASS_WEIGHTS.get(class_name, 1.00)
    for index, class_name in enumerate(CLASS_NAMES)
}
print('Class weights:', {CLASS_NAMES[index]: weight for index, weight in class_weight.items()})

model = tf.keras.models.load_model(E3_CHECKPOINT_PATH)
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=['accuracy'],
)

callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True),
    tf.keras.callbacks.ModelCheckpoint(CHECKPOINT_PATH, monitor='val_loss', save_best_only=True),
    tf.keras.callbacks.CSVLogger(HISTORY_PATH, append=False),
]

print('Starting from V2-E3. Maximum epochs: 12; early stopping may end sooner.')
start_time = time.time()
history = model.fit(
    train_ds, validation_data=validation_ds, epochs=12, callbacks=callbacks,
    class_weight=class_weight, verbose=1,
)
training_seconds = time.time() - start_time

training_summary = {
    'experiment': 'V2-E4',
    'starting_checkpoint': str(E3_CHECKPOINT_PATH),
    'target_class_weights': TARGET_CLASS_WEIGHTS,
    'learning_rate': 1e-5,
    'max_epochs': 12,
    'epochs_completed': len(history.history['loss']),
    'training_seconds': training_seconds,
    'best_validation_loss': float(min(history.history['val_loss'])),
    'best_validation_accuracy': float(max(history.history['val_accuracy'])),
}
SUMMARY_PATH.write_text(json.dumps(training_summary, indent=2), encoding='utf-8')
print(f'Training completed in {training_seconds:.1f} seconds.')
print('Best V2-E4 model saved in Drive:', CHECKPOINT_PATH)

In [ ]:
# Cell 6 — validate the saved best V2-E4 model. Rerun Cells 1–4 first if Colab disconnected.
if not CHECKPOINT_PATH.is_file():
    raise FileNotFoundError(f'No completed V2-E4 checkpoint found in Drive: {CHECKPOINT_PATH}')

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
best_model = tf.keras.models.load_model(CHECKPOINT_PATH)
y_validation = np.concatenate([labels.numpy() for _, labels in validation_ds])
probabilities = best_model.predict(validation_ds, verbose=1)
predictions = probabilities.argmax(axis=1)

precision, recall, f1_values, support = precision_recall_fscore_support(
    y_validation, predictions, labels=np.arange(len(CLASS_NAMES)), zero_division=0
)
e4_metrics = {
    'experiment': 'V2-E4',
    'split': 'validation',
    'protected_test_used': False,
    'image_count': int(len(y_validation)),
    'accuracy': float(accuracy_score(y_validation, predictions)),
    'macro_f1': float(f1_score(y_validation, predictions, average='macro')),
    'target_class_weights': TARGET_CLASS_WEIGHTS,
    'per_class': {
        class_name: {
            'precision': float(precision[index]),
            'recall': float(recall[index]),
            'f1': float(f1_values[index]),
            'support': int(support[index]),
        }
        for index, class_name in enumerate(CLASS_NAMES)
    },
}
(OUTPUT_DIR / 'v2_e4_validation_metrics.json').write_text(json.dumps(e4_metrics, indent=2), encoding='utf-8')

comparison = {
    'split': 'validation',
    'protected_test_used': False,
    'V2-E1_compact_cnn': {'accuracy': 0.6984318455971049, 'macro_f1': 0.6795904943261485},
    'V2-E2_frozen_mobilenetv2': {'accuracy': 0.8196622436670687, 'macro_f1': 0.8374032394969715},
    'V2-E3_finetuned_mobilenetv2': {'accuracy': 0.8552472858866104, 'macro_f1': 0.8704102628125944},
    'V2-E4_targeted_class_weights': {'accuracy': e4_metrics['accuracy'], 'macro_f1': e4_metrics['macro_f1']},
}
(OUTPUT_DIR / 'v2_e4_comparison.json').write_text(json.dumps(comparison, indent=2), encoding='utf-8')

report = classification_report(y_validation, predictions, target_names=CLASS_NAMES, digits=4, zero_division=0)
(OUTPUT_DIR / 'v2_e4_validation_classification_report.txt').write_text(report, encoding='utf-8')
print('Validation accuracy:', f"{e4_metrics['accuracy']:.4f}")
print('Validation macro F1:', f"{e4_metrics['macro_f1']:.4f}")
print(report)

matrix = confusion_matrix(y_validation, predictions, labels=np.arange(len(CLASS_NAMES)))
fig, axis = plt.subplots(figsize=(10, 8))
image = axis.imshow(matrix, cmap='Blues')
axis.set_xticks(np.arange(len(CLASS_NAMES)), CLASS_NAMES, rotation=45, ha='right')
axis.set_yticks(np.arange(len(CLASS_NAMES)), CLASS_NAMES)
axis.set_xlabel('Predicted label')
axis.set_ylabel('Actual label')
axis.set_title('V2-E4 validation confusion matrix')
threshold = matrix.max() / 2 if matrix.max() else 0
for row in range(matrix.shape[0]):
    for column in range(matrix.shape[1]):
        axis.text(column, row, str(matrix[row, column]), ha='center', va='center', color='white' if matrix[row, column] > threshold else 'black')
fig.colorbar(image, ax=axis)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'v2_e4_validation_confusion_matrix.png', dpi=150)
plt.show()

In [ ]:
# Cell 7 — create learning curves and a browser-download package. Do not commit the package or model.
if not HISTORY_PATH.is_file():
    raise FileNotFoundError(f'Missing V2-E4 training history in Drive: {HISTORY_PATH}')

history_frame = pd.read_csv(HISTORY_PATH)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history_frame['epoch'] + 1, history_frame['accuracy'], label='train')
axes[0].plot(history_frame['epoch'] + 1, history_frame['val_accuracy'], label='validation')
axes[0].set_title('V2-E4 accuracy')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[1].plot(history_frame['epoch'] + 1, history_frame['loss'], label='train')
axes[1].plot(history_frame['epoch'] + 1, history_frame['val_loss'], label='validation')
axes[1].set_title('V2-E4 loss')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'v2_e4_learning_curve.png', dpi=150)
plt.show()

for source_path in [CHECKPOINT_PATH, HISTORY_PATH, SUMMARY_PATH]:
    if source_path.is_file():
        shutil.copy2(source_path, OUTPUT_DIR / source_path.name)

archive_path = shutil.make_archive('/content/v2_e4_output', 'zip', OUTPUT_DIR)
print('Created browser-download package:', archive_path)
print('Keep this ZIP outside the Git repository. Do not commit it.')
files.download(archive_path)